# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Himanshu-rai-prog/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*


### Method choice

I will use Logistic Regression as the first learned model because the task has an observed yes/no label: whether a content item is declining.

The target is `is_declining_label`, defined as 1 when `trend_direction` is "down".

I will use a client-grouped train/test split so content from the same client does not appear in both training and test data. I will also exclude `trend_direction`, `trend_pct`, and the direct 30-day trend input columns to reduce target leakage.

For evaluation, I will compare the Week-4 rule baseline and the Logistic Regression model using precision@50 and precision@100 on the same held-out test set.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

RANDOM_STATE = 42

url = "https://raw.githubusercontent.com/Himanshu-rai-prog/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

print("Shape:", df.shape)
print("Loaded successfully.")
from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

RANDOM_STATE = 42

# Observed target
df["is_declining_label"] = (
    df["trend_direction"].eq("down").astype(int)
)

print("Target distribution:")
print(df["is_declining_label"].value_counts())
print("\nTarget rate:")
print(df["is_declining_label"].mean())

# Columns that can directly reveal the target
leakage_cols = [
    "trend_direction",
    "trend_pct",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
]

# IDs are used only for grouping
drop_cols = [
    "content_id",
    "client_id",
    "is_declining_label",
] + leakage_cols

X = df.drop(columns=drop_cols)
y = df["is_declining_label"]
groups = df["client_id"]

print("\nFeature shape:", X.shape)
print("Target shape:", y.shape)
print("Unique clients:", groups.nunique())

Shape: (30000, 44)
Loaded successfully.
Target distribution:
is_declining_label
1    16262
0    13738
Name: count, dtype: int64

Target rate:
0.5420666666666667

Feature shape: (30000, 34)
Target shape: (30000,)
Unique clients: 32


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

### Split design

I will use a client-grouped train/test split. This prevents content from the same client appearing in both training and test sets, which helps reduce client-level leakage.

I will use 80% of the data for training and 20% for testing, with a fixed random seed for reproducibility.

The target distribution will be checked in both sets to make sure the split remains reasonably representative.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.model_selection import GroupShuffleSplit

RANDOM_STATE = 42

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

groups_train = groups.iloc[train_idx].copy()
groups_test = groups.iloc[test_idx].copy()

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

print("\nTrain target rate:", y_train.mean())
print("Test target rate:", y_test.mean())

print("\nTrain clients:", groups_train.nunique())
print("Test clients:", groups_test.nunique())

print("\nShared clients:",
      len(set(groups_train).intersection(set(groups_test))))

Train shape: (23837, 34)
Test shape: (6163, 34)

Train target rate: 0.5501111717078492
Test target rate: 0.5109524582184002

Train clients: 25
Test clients: 7

Shared clients: 0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

### Train the Logistic Regression model

I will train Logistic Regression on the client-grouped training set.

The model will output a probability that a content item is declining. I will use this probability to rank test items from highest to lowest predicted decline risk.

I will then compare this ranking with the Week-4 baseline rule using precision@50 and precision@100 on the same test set.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Train Logistic Regression

numeric_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]),
            numeric_features
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore"))
            ]),
            categorical_features
        )
    ]
)

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=RANDOM_STATE
    ))
])

model.fit(X_train, y_train)

model_prob = model.predict_proba(X_test)[:, 1]

print("Model trained successfully.")
print("Predictions:", len(model_prob))
# Week-4 baseline rule on the same test set

test_eval = df.iloc[test_idx].copy()

test_eval["baseline_score"] = np.where(
    (test_eval["days_since_last_update"] >= 181) &
    (test_eval["impressions_90d"] >= 500),
    test_eval["impressions_90d"],
    0
)

test_eval["model_score"] = model_prob
test_eval["actual"] = y_test.values

def precision_at_k(data, score_col, k):
    ranked = data.sort_values(score_col, ascending=False).head(k)
    return ranked["actual"].mean()

results = []

for k in [50, 100]:
    results.append({
        "k": k,
        "baseline_precision": precision_at_k(
            test_eval, "baseline_score", k
        ),
        "model_precision": precision_at_k(
            test_eval, "model_score", k
        )
    })

comparison = pd.DataFrame(results)

print(comparison)

Model trained successfully.
Predictions: 6163
     k  baseline_precision  model_precision
0   50                0.52             0.64
1  100                0.56             0.66


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

### Errors and interpretation

The Logistic Regression model produced higher precision@50 and precision@100 than the Week-4 rule baseline on the same client-grouped test set.

This is an observed result on this dataset and split, not evidence that the model will generalize to every future dataset.

To understand model behavior, I will inspect the largest false-positive and false-negative cases and review the strongest model coefficients.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Error analysis

test_eval["predicted"] = (test_eval["model_score"] >= 0.5).astype(int)

false_positives = test_eval[
    (test_eval["predicted"] == 1) &
    (test_eval["actual"] == 0)
].copy()

false_negatives = test_eval[
    (test_eval["predicted"] == 0) &
    (test_eval["actual"] == 1)
].copy()

print("False positives:", len(false_positives))
print("False negatives:", len(false_negatives))

print("\nExample false positives:")
print(
    false_positives[
        ["content_id", "model_score", "actual"]
    ].head(3)
)

print("\nExample false negatives:")
print(
    false_negatives[
        ["content_id", "model_score", "actual"]
    ].head(3)
)
# Top model features

feature_names = model.named_steps["preprocessor"].get_feature_names_out()
coefficients = model.named_steps["classifier"].coef_[0]

feature_importance = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefficients
})

feature_importance["abs_coefficient"] = (
    feature_importance["coefficient"].abs()
)

top_features = feature_importance.sort_values(
    "abs_coefficient",
    ascending=False
).head(10)

print("Top 10 model features:")
print(top_features[["feature", "coefficient"]])
print("ML-08 self-check complete.")
print("Train rows:", len(X_train))
print("Test rows:", len(X_test))
print("Precision@50 - Baseline:", comparison.loc[comparison["k"] == 50, "baseline_precision"].iloc[0])
print("Precision@50 - Model:", comparison.loc[comparison["k"] == 50, "model_precision"].iloc[0])
print("Precision@100 - Baseline:", comparison.loc[comparison["k"] == 100, "baseline_precision"].iloc[0])
print("Precision@100 - Model:", comparison.loc[comparison["k"] == 100, "model_precision"].iloc[0])

False positives: 1603
False negatives: 1097

Example false positives:
              content_id  model_score  actual
13  content_a5a2fbc76336     0.759039       0
26  content_72c5c2d73e5a     0.583816       0
36  content_bce275871a25     0.679296       0

Example false negatives:
              content_id  model_score  actual
23  content_2da6ae9d0882     0.304048       1
39  content_4595e8704e07     0.313364       1
43  content_1938955b34c4     0.234657       1
Top 10 model features:
                              feature  coefficient
64           cat__position_tier_top_3    -1.028890
9                      num__users_90d    -0.916635
8                   num__sessions_90d     0.799593
13         num__days_with_impressions     0.781887
38         cat__model_used_gpt-5-mini     0.585161
27   cat__content_type_feedly article    -0.538438
39            cat__model_used_unknown    -0.537943
28  cat__content_type_keyword article     0.517130
14            num__days_with_sessions    -0.500916
31 

## Self-check

Before you submit, confirm each line honestly:

- [ x] Every section above is filled — markdown thinking AND the code that backs it
- [ x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ x] No client names, URLs, or private queries anywhere
- [x ] My claims use careful words: observed, measured, directional, decision-support
- [x ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.